# Qwen2.5-VL Document OCR Server on Kaggle GPU
Máy chủ bóc tách tài liệu (Bảng biểu, văn bản, công thức toán) sang định dạng Markdown chuẩn.
Kết nối với hệ thống Web App qua Cloudflare Tunnel.

In [ ]:
# 1. Don dep torchaudio va cai dat thu vien
!pip uninstall -y torchaudio -q
!pip install -q --no-warn-conflicts --upgrade "transformers>=4.49.0" accelerate qwen-vl-utils pycloudflared fastapi uvicorn torchvision nest_asyncio


In [ ]:
# 2. Kiểm tra GPU và định vị trọng số mô hình
import os
import sys
import glob
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

# Tự động tìm kiếm model được mount từ Kaggle Models / Datasets
config_files = glob.glob("/kaggle/input/**/config.json", recursive=True)
model_path = None
for cfg in config_files:
    if any(k in cfg.lower() for k in ["qwen", "vl"]):
        model_path = os.path.dirname(cfg)
        break

if model_path:
    print(f"\n[THƯ MỤC CÓ SẴN] Phát hiện trọng số mô hình mount sẵn tại: {model_path}")
    print("Nạp 16.6GB trọng số trực tiếp từ ổ đĩa (0 giây tải)!")
else:
    model_path = "Qwen/Qwen2.5-VL-7B-Instruct"
    print(f"\n[ONLINE] Tải từ Hugging Face: {model_path}")

In [ ]:
# 3. Nạp mô hình Qwen2.5-VL lên GPU
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

print(f"Đang nạp mô hình {model_path} lên GPU...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Nạp AutoProcessor với cơ chế fallback tự động
try:
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
except Exception as e:
    print(f"Preprocessor cục bộ thiếu cấu hình ({e}), đang nạp cấu hình chuẩn từ Hugging Face...")
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", trust_remote_code=True)

print("Nạp mô hình và bộ xử lý thành công!")

In [ ]:
# 4. Khoi tao FastAPI Server va Cloudflare Tunnel
import io
import time
import base64
import threading
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
from qwen_vl_utils import process_vision_info

app = FastAPI(title="Kaggle Qwen2.5-VL OCR API")

class OCRRequest(BaseModel):
    image_base64: str
    prompt: str = (
        "Ban la chuyen gia OCR tai lieu cao cap. Hay chuyen doi hinh anh tai lieu nay "
        "thanh dinh dang Markdown chuan xac. Yeu cau bat buoc:\n"
        "1. Giu nguyen 100% toan bo bang bieu so lieu bang dinh dang Markdown Table (| Cot 1 | Cot 2 |).\n"
        "2. Giu nguyen cac cap tieu de (#, ##, ###), danh sach liet ke, va thu tu doc cua van ban nhieu cot.\n"
        "3. Bao toan chinh xac toan bo chu tieng Viet co dau va cac ky tu dac biet.\n"
        "4. Tuyet doi khong them loi giai thich, loi chao hay binh luan; chi xuat duy nhat noi dung Markdown cua tai lieu."
    )

class OCRResponse(BaseModel):
    markdown: str
    inference_time_seconds: float

@app.get("/health")
def health():
    return {"status": "healthy", "device": "cuda" if torch.cuda.is_available() else "cpu"}

@app.post("/ocr", response_model=OCRResponse)
def perform_ocr(req: OCRRequest):
    t0 = time.time()
    try:
        image_bytes = base64.b64decode(req.image_base64)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Khong the doc anh: {str(e)}")

    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": req.prompt}
                ]
            }
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=2048)
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]

        elapsed = round(time.time() - t0, 2)
        return OCRResponse(markdown=output_text.strip(), inference_time_seconds=elapsed)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Loi suy luan: {str(e)}")

# Mo Cloudflare Tunnel
port = 8000
tunnel = try_cloudflare(port=port)
public_url = getattr(tunnel, "tunnel", str(tunnel))

print("\n" + "=" * 70)
print("MAY CHU KAGGLE OCR DA SAN SANG HOAT DONG!")
print(f"URL Endpoint: {public_url}/ocr")
print(f"URL Health:   {public_url}/health")
print("=" * 70)
print("Sao chep URL Endpoint tren va dan vao trang Admin / Settings!\n")

# Chay uvicorn trong Thread doc lap de tranh xung dot asyncio loop cua Jupyter
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=port)

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Duy tri cell chay lien tuc de phuc vu request
while True:
    time.sleep(30)
